# Define, visualize, and customize a simulation plan

The first three sections generate two plans without loading a dataset or training a model. Six logical clients compete for three active assignments, and each accepted upload publishes a model. The second plan slows only `client-0` uploads, so we can inspect the resulting timeline and staleness. The last section replays one plan with a small NumPy FedBuff job.

From an NVFlare source checkout, install the notebook dependencies in the Python environment you want to use as its kernel:

```bash
python -m pip install -e .
python -m pip install numpy matplotlib ipykernel notebook
python -m ipykernel install --user --name nvflare-plan --display-name "NVFlare plan"
cd research/fedrevive
jupyter notebook simulation_plan_tutorial.ipynb
```

Select the **NVFlare plan** kernel, then choose **Run > Run All Cells**. Run from `research/fedrevive` so the planner and the existing NumPy example can be imported. The linked figures below are committed references for the supplied seed, settings, and planner code; rerunning produces the same timelines. After changing a delay, open the newly generated PNGs in `plan_demo/` to see that run. The cells save JSON plans, PNG plots, and replay artifacts there. All durations are **virtual time units**, not measured wall-clock seconds.

In [1]:
import json
import sys
from pathlib import Path

from nvflare.apis.simulation_plan import validate_simulation_plan

research_dir = Path.cwd().resolve()
if not (research_dir / "schedule_standard.py").is_file():
    raise FileNotFoundError("Start the notebook from research/fedrevive")
sys.path.insert(0, str(research_dir))

from plot_plan import plot_simulation_plan
from schedule_standard import make_simulation_plan, sample_paper_delay

output = Path("plan_demo")
output.mkdir(exist_ok=True)


## 1. Define the baseline

Each client has a fixed download duration, exponentially sampled training duration, and uniformly sampled upload duration. The callback uses the planner's assignment- and phase-specific RNG. Both plans below use that callback path so their samples are comparable.

In [2]:
profiles = {
    f"client-{i}": {
        "download_mean": 0.10,
        "train_mean": 0.70,
        "upload_mean": 0.15,
        "upload_half_width": 0.02,
    }
    for i in range(6)
}
settings = dict(
    run_seed=10,
    max_time=20.0,
    num_rounds=12,
    concurrency=3,
    min_open_slots=1,
    buffer_size=1,
)


def baseline_delay(assignment_id, logical_name, phase, profile, rng):
    return sample_paper_delay(phase, profile, rng)


baseline = validate_simulation_plan(make_simulation_plan(profiles, **settings, delay_sampler=baseline_delay))
(output / "baseline.json").write_text(json.dumps(baseline, indent=2) + "\n", encoding="utf-8")
figure = plot_simulation_plan(baseline, limit=14)
figure.savefig(output / "baseline.png", dpi=150)


**Reference plot:** Baseline plan reference timeline.

![Baseline plan reference timeline](figs/plan_notebook_baseline.png)

## 2. Change one delay rule

Multiply `client-0`'s upload duration by 20 and regenerate the plan. Change `upload_multiplier` and rerun this cell to explore other delays. The planner computes new reservations and model bases; a changed delay can affect later client selection too.

In [3]:
upload_multiplier = 20.0


def slow_upload(assignment_id, logical_name, phase, profile, rng):
    duration = baseline_delay(assignment_id, logical_name, phase, profile, rng)
    return duration * upload_multiplier if logical_name == "client-0" and phase == "upload" else duration


custom = validate_simulation_plan(make_simulation_plan(profiles, **settings, delay_sampler=slow_upload))
(output / "slow_upload.json").write_text(json.dumps(custom, indent=2) + "\n", encoding="utf-8")
figure = plot_simulation_plan(custom, limit=14)
figure.savefig(output / "slow_upload.png", dpi=150)


**Reference plot:** Slower client-0 upload reference timeline.

![Slower client-0 upload reference timeline](figs/plan_notebook_slow_upload.png)

## 3. Compare acceptance and staleness

For this `buffer_size=1` example, staleness at acceptance is the number of earlier accepted uploads minus the assignment's base version. For larger buffers, count **published versions** instead of uploads. With the default multiplier, the first `client-0` upload moves from acceptance **#1, staleness 0** to **#7, staleness 6**.

In [4]:
def first_client_zero_upload(plan):
    execution = plan["async_execution"]
    uploads = [
        event
        for event in execution["timing"]["events"]
        if event["phase"] == "upload" and event["virtual_time"] < execution["timing"]["horizon"]
    ]
    for accepted_before, event in enumerate(uploads):
        assignment_id = event["assignment_id"]
        if execution["logical_sequence"][assignment_id] == "client-0":
            return accepted_before + 1, accepted_before - execution["base_versions"][assignment_id]
    raise ValueError("client-0 has no accepted upload in this plan")


for label, plan in (("baseline", baseline), ("slow upload", custom)):
    rank, staleness = first_client_zero_upload(plan)
    print(f"{label}: first client-0 upload accepted #{rank}, staleness {staleness}")


baseline: first client-0 upload accepted #1, staleness 0
slow upload: first client-0 upload accepted #7, staleness 6


## 4. Replay a saved plan

Run the next cell to replay `slow_upload.json` with the dataset-free components from the existing NumPy FedBuff example. The recipe uses the same publication and scheduling settings as the plan. Its `min_clients=2` and SimEnv's `num_clients=2` refer to physical worker sites; the plan controls six logical clients. SimEnv preflights the plan before training and verifies the recorded logical events afterward.

See [FedRevive replay setup](README.md#replay-a-simulator-plan) for a runnable recipe command, the [viewer options](README.md#visualize-a-plan-before-running-it) for later assignment windows and peak storage estimates, and the [simulation plan guide](../../docs/user_guide/data_scientist_guide/reproducible_simulation.rst) for plan validation and trace files.

In [ ]:
example_package = research_dir.parents[1] / "examples" / "advanced" / "collab" / "numpy_async_reproducible"
if not (example_package / "job.py").is_file():
    raise FileNotFoundError("Run this notebook from an NVFlare source checkout")
sys.path.insert(0, str(example_package.parent))

from numpy_async_reproducible.job import FinalWeightsObserver, NumpyModelLoader, SeededNumpyClient
from numpy_async_reproducible.model import NumpyModelOps
from nvflare.collab import FedBuffRecipe
from nvflare.recipe import SimEnv

recipe = FedBuffRecipe(
    name="simulation_plan_notebook",
    client=SeededNumpyClient(),
    model_loader=NumpyModelLoader(),
    model_ops=NumpyModelOps(),
    observer=FinalWeightsObserver(),
    num_rounds=settings["num_rounds"],
    concurrency=settings["concurrency"],
    buffer_size=settings["buffer_size"],
    min_open_slots=settings["min_open_slots"],
    min_clients=2,
    call_timeout=30,
)
recipe.set_client_prop("run_seed", settings["run_seed"])
recipe.add_server_file(str(example_package))
recipe.add_client_file(str(example_package))
run = recipe.execute(
    SimEnv(
        num_clients=2,
        workspace_root=str((output / "replay_workspace").resolve()),
        reprod=True,
        plan=str((output / "slow_upload.json").resolve()),
    )
)
print("Job status:", run.get_status())
print("Results:", run.get_result(clean_up=False))
